# 03 · Deployment-time certificates: first experiment (NF-UNSW-NB15-v3 and NF-ToN-IoT-v3)

What this notebook tests. For each flow the detector calls benign, the certificate asks: could an attacker have produced this flow from a flow the detector would have flagged, using only the declared operations within budget? Three outcomes:
- **CERTIFIED**: a proof that no such original flow exists.
- **WITNESS**: a concrete original flow that the detector flags and that the operations could have turned into the observed flow. It is checked with the real model and against the operation model before it counts. This is the explanation an analyst gets.
- **UNRESOLVED**: no proof and no checked witness within 1 second. The flow is treated as not certified.

Design decisions, fixed on 30 Sep 2026 before any certificate result:
- Attacker: the sending side of the flow, changing only its own packets. Operations: padding each packet, adding dummy packets (40 to 1514 bytes), delaying packets.
- Primary budget (pass/fail): up to 100 bytes of padding per packet (never beyond 1514), up to 5 dummy packets, up to 1,000 ms of added delay. Sensitivity only: low (20 bytes, 1 packet, 100 ms) and high (400 bytes, 20 packets, 5,000 ms).
- The operation model uses only the relations that notebook 02 found to hold on at least 99.9% of rows in all three datasets. Fields with no audited formula (per-second bytes, inter-arrival statistics) are left free whenever the operations that could affect them are used. This keeps the certificate sound at the cost of certifying less.
- Every simplification widens the set of possible original flows, so a certificate is never issued when a flagged original exists inside the model. A flow that itself breaks an audited relation is out of model and counted as UNRESOLVED.
- Time limit: 1 second per flow in total (fast bound, witness search and exact solver together). An answer that arrives later is recorded but counted as UNRESOLVED. One worker per physical core, so each check has a whole core; the machine is recorded.
- Flows under test: 5,000 clean benign flows (label benign, detector says benign) per dataset from the development role, drawn with a fixed seed. Calibration and locked_test are not used.
- Pre-registered pass rules, per dataset: at least 90% of clean benign flows CERTIFIED, and at most 5% UNRESOLVED. The third rule (fewer deferrals than an invariant-feature gate at equal evasion catch) is evaluated in notebook 04.
- Soundness test on real flows: flagged attack flows from development get random in-budget operations (40 tries each, the lowest-scoring kept). Every result the detector calls benign is an evasion, and the certificate must never certify one.
- Also reported, not scored: missed attacks (attack flows the detector already calls benign) and the low and high budgets on the first 1,000 flows of the same sample.
- NF-CSE-CIC-IDS2018-v3 is not run here; it joins at the confirmation stage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import gc, hashlib, importlib, json, os, sys, time
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/XIDS_Certify_Research')
REPO = PROJECT / 'xids-certify'
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import scipy
import xgboost as xgb
import src.nf3_data as nf
import src.detector as det
import src.operations as ops
import src.certify as cf
for m in (nf, det, ops, cf):
    importlib.reload(m)

TABLES = REPO / 'results' / 'tables'
CACHE = PROJECT / 'cache' / 'nf03'
CACHE.mkdir(parents=True, exist_ok=True)
DATASETS = ['unsw', 'ton']
N_BENIGN, N_SENS, N_MISSED, N_SOUND, TRIES = 5000, 1000, 1000, 2000, 40
TIME_LIMIT = 1.0
SEED = 20260930
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 120)

ops_text = json.dumps(ops.OPERATIONS_CONFIG, indent=2, sort_keys=True)
(REPO / 'configs' / 'operations_v1.json').write_text(ops_text + '\n')
OPS_SHA = hashlib.sha256(ops_text.encode()).hexdigest()
MODELS = json.loads((TABLES / 'nf02_models.json').read_text())
MACHINE = cf.machine()
VERSIONS = {'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__, 'xgboost': xgb.__version__}
print('machine:', MACHINE)
print('versions:', VERSIONS)
print('operations config sha256:', OPS_SHA[:16], '| budgets:', ops.BUDGETS)

## Detectors, thresholds and the flows under test

In [ ]:
S = {}
rng = np.random.default_rng(SEED)
for ds in DATASETS:
    rec = MODELS[ds]
    path = PROJECT / rec['model_path']
    assert hashlib.sha256(path.read_bytes()).hexdigest() == rec['model_sha256'], f'{ds}: model file changed'
    booster = xgb.Booster()
    booster.load_model(str(path))
    feats = rec['features']
    ens = cf.Ensemble(booster, feats)                       # checks the parser against the booster itself
    train = nf.load_split(PROJECT, ds, roles=['detector_train'], columns=feats)
    integer = ops.integer_features(train.iloc[:400_000], feats)
    del train
    dev = nf.load_split(PROJECT, ds, roles=['development'], columns=feats + ['row_idx', 'y', 'family', 'multiplicity'])
    X = dev[feats].to_numpy(np.float32)
    score = det.predict(booster, dev[feats])
    thr = rec['threshold']
    benign_verdict = score < thr
    y = dev['y'].to_numpy()
    pick = lambda mask, n: np.sort(rng.permutation(np.flatnonzero(mask))[:n])
    S[ds] = {'ens': ens, 'thr': thr, 'integer': integer, 'feats': feats, 'X': X, 'dev': dev, 'score': score,
             'clean': pick(benign_verdict & (y == 0), N_BENIGN), 'missed': pick(benign_verdict & (y == 1), N_MISSED),
             'flagged': pick(~benign_verdict & (y == 1), N_SOUND)}
    print(f"{ds}: threshold {thr:.6g} | trees {len(ens.trees)} | integer features {len(integer)}/{len(feats)} | "
          f"development rows {len(dev):,} | clean benign sampled {len(S[ds]['clean']):,} | "
          f"missed attacks {int((benign_verdict & (y == 1)).sum()):,} (sampled {len(S[ds]['missed']):,}) | "
          f"flagged attacks sampled {len(S[ds]['flagged']):,}")
    gc.collect()

## Soundness test on real flows: in-budget evasions must never be certified

In [ ]:
SOUND = {}
for ds in DATASETS:
    st = S[ds]
    ens, feats, X = st['ens'], st['feats'], st['X']
    m_thr = cf.logit(st['thr'])
    r2 = np.random.default_rng(SEED + 1)
    ev_rows, ev_ids, attacked = [], [], 0
    for i in st['flagged']:
        z = dict(zip(feats, map(float, X[i])))
        tries = [ops.apply_operations(z, ops.BUDGETS['primary'], r2) for _ in range(TRIES)]
        cands = np.stack([np.array([t[f] for f in feats], dtype=np.float32) for t in tries])
        m = ens.margin(cands)
        j = int(np.argmin(m))
        attacked += 1
        if m[j] < m_thr:
            ev_rows.append(cands[j])
            ev_ids.append(int(i))
    print(f'{ds}: {attacked:,} flagged attack flows attacked, {len(ev_ids):,} evasions found ({len(ev_ids)/max(attacked,1):.1%})')
    if ev_ids:
        V = cf.Verifier(ens, st['thr'], st['integer'], ops.BUDGETS['primary'])
        res = cf.run(V, np.stack(ev_rows), ev_ids, CACHE / f'{ds}_soundness.parquet', TIME_LIMIT)
        SOUND[ds] = cf.summarise(res, {'dataset': ds, 'test': 'soundness', 'evasions': len(ev_ids)})
        assert SOUND[ds]['certified'] == 0, f'{ds}: SOUNDNESS FAILURE, an in-budget evasion was certified'
    else:
        SOUND[ds] = {'dataset': ds, 'test': 'soundness', 'evasions': 0}
display(pd.DataFrame(SOUND.values()))

## Primary budget on clean benign flows (pass/fail)

In [ ]:
RESULTS, ROWS = {}, []

def certify(ds, which, budget, n=None):
    st = S[ds]
    ids = st[which][:n] if n else st[which]
    V = cf.Verifier(st['ens'], st['thr'], st['integer'], ops.BUDGETS[budget])
    res = cf.run(V, st['X'][ids], ids, CACHE / f'{ds}_{which}_{budget}.parquet', TIME_LIMIT)
    RESULTS[(ds, which, budget)] = res
    row = cf.summarise(res, {'dataset': ds, 'flows_tested': which, 'budget': budget})
    ROWS.append(row)
    return row

for ds in DATASETS:
    print(ds, json.dumps(certify(ds, 'clean', 'primary'), indent=1))

## Missed attacks: attack flows the detector already calls benign (reported, not scored)

In [ ]:
for ds in DATASETS:
    if len(S[ds]['missed']):
        print(ds, json.dumps(certify(ds, 'missed', 'primary'), indent=1))

## Sensitivity: low and high budgets on the first 1,000 clean benign flows (reported, not scored)

In [ ]:
for ds in DATASETS:
    for budget in ('low', 'high'):
        print(ds, budget, json.dumps(certify(ds, 'clean', budget, N_SENS), indent=1))

## What blocks certification

For every WITNESS, the fields that differ between the observed flow and its flagged original show what the attacker would have had to change. Counting them across witnesses shows which fields stop the detector's benign verdicts from being certified.

In [ ]:
BLOCK, EXAMPLES = [], []
for ds in DATASETS:
    res = RESULTS[(ds, 'clean', 'primary')]
    w = res[res['status'] == 'WITNESS']
    counts = {}
    for c in w['changed']:
        for f in json.loads(c):
            counts[f] = counts.get(f, 0) + 1
    for f, n in sorted(counts.items(), key=lambda t: -t[1]):
        BLOCK.append({'dataset': ds, 'feature': f, 'witnesses_changing_it': n, 'share_of_witnesses': n / max(len(w), 1)})
    for _, r in w.sort_values('n_changed').head(3).iterrows():
        EXAMPLES.append({'dataset': ds, 'row_idx': r['row_idx'], 'fields_changed': r['n_changed'],
                         'dummy_pkts': r.get('op_dummy_pkts'), 'padding_bytes_removed': r.get('op_padding_bytes'),
                         'timing_changed': r.get('op_timing_changed'), 'observed -> flagged original': r['changed']})
block = pd.DataFrame(BLOCK)
for ds in DATASETS:
    display(block[block['dataset'] == ds].head(12))
display(pd.DataFrame(EXAMPLES))

## Pre-registered decision

In [ ]:
summary = pd.DataFrame(ROWS)
primary = summary[(summary['flows_tested'] == 'clean') & (summary['budget'] == 'primary')].copy()
primary['rule1_certified_ge_90pct'] = primary['certified_share'] >= 0.90
primary['rule2_unresolved_le_5pct'] = primary['unresolved_share'] <= 0.05
primary['passes_rules_1_and_2'] = primary['rule1_certified_ge_90pct'] & primary['rule2_unresolved_le_5pct']
display(primary[['dataset', 'flows', 'certified_share', 'witness_share', 'unresolved_share', 'out_of_model', 'over_time',
                 'median_seconds', 'p95_seconds', 'rule1_certified_ge_90pct', 'rule2_unresolved_le_5pct',
                 'passes_rules_1_and_2']])
summary.to_csv(TABLES / 'nf03_certificate_summary.csv', index=False)
pd.DataFrame(SOUND.values()).to_csv(TABLES / 'nf03_soundness.csv', index=False)
block.to_csv(TABLES / 'nf03_blocking_fields.csv', index=False)
pd.DataFrame(EXAMPLES).to_csv(TABLES / 'nf03_witness_examples.csv', index=False)
for (ds, which, budget), res in RESULTS.items():
    res.to_csv(TABLES / f'nf03_flows_{ds}_{which}_{budget}.csv.gz', index=False)
out = {'operations_config_sha256': OPS_SHA, 'machine': MACHINE, 'versions': VERSIONS, 'time_limit_seconds': TIME_LIMIT,
       'seed': SEED, 'detector_models': {d: MODELS[d]['model_sha256'] for d in DATASETS},
       'decision': {r['dataset']: {'certified_share': r['certified_share'], 'unresolved_share': r['unresolved_share'],
                                   'passes_rules_1_and_2': bool(r['passes_rules_1_and_2'])} for _, r in primary.iterrows()}}
(TABLES / 'nf03_summary.json').write_text(json.dumps(out, indent=2) + '\n')
print(json.dumps(out['decision'], indent=2))

In [ ]:
import os, subprocess
os.chdir("/content/drive/MyDrive/XIDS_Certify_Research/xids-certify")
msg = "Notebook 03: first certificate experiment on NF-UNSW-NB15-v3 and NF-ToN-IoT-v3 with the pre-registered primary budget (100-byte padding, 5 dummy packets, 1,000 ms delay) and 1 s per flow; soundness test on real in-budget evasions; missed attacks and low/high budgets reported; fields that block certification; decision on pass rules 1 and 2"
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)